# Dedekind DSL: Analyst Tier
Declarative table workflow with built-in data-quality cleanup for messy real-world inputs.

> **See also**: [Splink](https://github.com/moj-analytical-services/splink) (MoJ) — probabilistic record linkage / entity resolution at scale (Fellegi-Sunter + EM).
> Splink and dedekind are complementary: the quality combinators here produce the standardised input that Splink requires,
> and Splink's output `cluster_id` is a natural panel key for a subsequent `smart_join`.
> The match ladder in `smart_join` is designed to accommodate Splink-style probabilistic scoring as a future higher tier (see [#253](https://github.com/vincentk/dedekind/issues/253)).

In [1]:
import pandas as pd

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None

from dedekind import (
    analyst_sales_quality_lift_report,
    monthly_category_report,
    smart_join,
    table,
)

print("Imported Analyst middleware: report builders, quality labels, and smart joins")
if plt is None:
    print("matplotlib not available: chart cells will print guidance instead of rendering charts")

ModuleNotFoundError: No module named 'dedekind'

## Part 1: Messy Source Tables
Start from intentionally noisy inputs (mixed case, symbols, missing values, malformed numbers).

In [2]:
df_sales = pd.DataFrame(
    [
        {"date": "2026-01-05", "product_id": " P1 ", "region": "North", "units": "10", "revenue": "100"},
        {"date": "2026-01-07", "product_id": "p2", "region": "NORTH", "units": "5", "revenue": "$250"},
        {"date": "2026-01-09", "product_id": "p3", "region": "south", "units": 7, "revenue": "140.0"},
        {"date": "2026-02-03", "product_id": "p1", "region": " SOUTH ", "units": 8, "revenue": "80"},
        {"date": "2026-02-10", "product_id": "p2", "region": "north", "units": "4x", "revenue": "200"},
        {"date": "2026-02-11", "product_id": "p3", "region": "south", "units": 10, "revenue": "oops"},
    ]
)

df_products = pd.DataFrame(
    [
        {"product_id": "p1", "category": " hardware "},
        {"product_id": "p2", "category": "SOFTWARE"},
        {"product_id": "p3", "category": "Accessories"},
    ]
)

df_regions = pd.DataFrame(
    [
        {"region": "north", "segment": "Enterprise"},
        {"region": "south", "segment": "SMB"},
    ]
)

sales = table("sales", df_sales)
products = table("products", df_products)
regions = table("regions", df_regions)

display(df_sales.head())

NameError: name 'table' is not defined

## Part 2: Clean, Join, And Aggregate
Apply quality combinators before business transformations, then build the monthly category report.

Econometrics note: this join + month derivation step is effectively building a panel (entity keys observed across time periods).

In [3]:
sales_clean = (
    sales
    .normalize_text(["product_id", "region"], lower=True, strip=True)
    .coerce_numeric(["units", "revenue"], strip_symbols=True)
    .fill_missing(units=0, revenue=0)
    .expect_domain("region", ["north", "south"], on_fail="raise")
)

products_clean = (
    products
    .normalize_text(["product_id", "category"], lower=True, strip=True)
    .expect_domain(
        "category",
        ["hardware", "software", "accessories"],
        on_fail="coerce",
        unknown_value="unknown",
    )
)

regions_clean = regions.normalize_text(["region"], lower=True, strip=True)

report_plan = monthly_category_report(
    smart_join(sales_clean, products_clean).smart_join(regions_clean),
)

summary = report_plan.realize()

print("logical plan")
print(report_plan.explain())

print("\nsummary")
display(summary)

NameError: name 'sales' is not defined

## Part 3: Error Analysis - Quality Lift From Additional Trusted Tables
We compare a vanilla pipeline against a smart pipeline and report quality-lift deltas as consumable data products from middleware.

Analyst facade assumptions in this demo:
- `smart_join` works in best-effort mode by default, and can later accept trust hints for selected columns/ranges.
- `smart_pivot` uses sensible defaults by default, and can later accept user-interest hints.
- Extra rows can still improve inference/scaffolding even when they are not pivoted directly.

**Trusted-target semantics:**
A _trusted table_ encodes a structural prior about what records *should* exist.
Example: the analyst knows the dataset must contain exactly one record per day per region.
By joining the messy source against that trusted skeleton, the pipeline can:
- detect *gaps* — days/regions expected but absent in the source;
- flag *duplicates* — source rows matching the same skeleton slot more than once;
- *bootstrap error estimates* from a known structural prior rather than purely from observed-data statistics.

This is the mechanism by which extra high-quality reference tables (the `product_price_hq` table below) improve quality labels in step (d) relative to the vanilla pipeline in step (c).

In [ ]:
# Reference (trusted) target for error analysis
reference_pivot = pd.DataFrame(
    [
        {"month": "2026-01", "accessories": 140.0, "hardware": 100.0, "software": 250.0},
        {"month": "2026-02", "accessories": 200.0, "hardware": 80.0, "software": 200.0},
    ]
)

# Additional high-quality reference table (trusted source)
df_product_price_hq = pd.DataFrame(
    [
        {"product_id": "p1", "unit_price_hq": 10.0},
        {"product_id": "p2", "unit_price_hq": 50.0},
        {"product_id": "p3", "unit_price_hq": 20.0},
    ]
)
product_price_hq = table("product_price_hq", df_product_price_hq)

analyst_report = analyst_sales_quality_lift_report(
    sales,
    products,
    regions,
    reference_pivot=reference_pivot,
    product_price_hq=product_price_hq,
)

input_quality = analyst_report["input_quality"]
vanilla_check = analyst_report["vanilla_pivot"]
smart_check = analyst_report["smart_pivot"]
error_report = analyst_report["error_report"]
quality_labels = analyst_report["quality_labels"]
quality_delta = analyst_report["quality_delta"]
quality_issues = analyst_report["sales_quality_issues"]
region_values = analyst_report["region_values"]

print("a) dirty table A (sales): quality metrics and certifications")
display(input_quality[input_quality["table"] == "table_a_sales"])

print("b) dirty table B (products): quality metrics and certifications")
display(input_quality[input_quality["table"] == "table_b_products"])

print("c) vanilla join + pivot with estimated quality labels")
display(vanilla_check)
display(quality_labels[quality_labels["stage"] == "vanilla"])

print("d) smart join + smart pivot with improved quality labels")
display(smart_check)
display(quality_labels[quality_labels["stage"] == "smart"])

print("e) quality-lift delta (vanilla -> smart)")
display(quality_delta)

print("sales quality interventions")
display(quality_issues)

print("error report (lower is better)")
display(error_report)

if plt is not None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)

    vanilla_check.set_index("month").plot(kind="bar", ax=axes[0], title="Vanilla Pivot")
    smart_check.set_index("month").plot(kind="bar", ax=axes[1], title="Smart Pivot")

    axes[0].set_ylabel("Revenue")
    axes[1].set_ylabel("Revenue")
    axes[0].legend(title="Category")
    axes[1].legend(title="Category")

    error_report.plot(x="stage", y="pivot_mae", kind="bar", legend=False, ax=axes[2], title="Pivot Error By Stage")
    axes[2].set_ylabel("MAE vs Reference")
    axes[2].tick_params(axis="x", rotation=30)

    plt.show()
else:
    print("Install matplotlib to render charts: pip install matplotlib")

print("realized region values:", region_values)

broken-data pivot table


category,month,accessories,hardware,software
0,2026-01,0.0,0.0,0.0
1,2026-02,0.0,0.0,0.0


correlation-imputed pivot table


category,month,accessories,hardware,software
0,2026-01,140.0,100.0,250.0
1,2026-02,200.0,80.0,200.0


extra-table-enriched pivot table


...

quality interventions


,issue_type,column,count,step,relation,details
0,numeric_coercion_loss,units,1,2,sales,errors=coerce
1,numeric_coercion_loss,revenue,1,2,sales,errors=coerce
2,filled_missing,units,1,3,sales,fill=0
3,filled_missing,revenue,1,3,sales,fill=0


error reduction report (lower is better)


,stage,pivot_mae
0,broken,161.666667
1,gap_fill_only,33.333333
2,corr_imputed,0.000000
3,extra_hq_table,0.000000
4,cleaned_pipeline,33.333333


Install matplotlib to render charts: pip install matplotlib
realized region values: ['north', 'south']


In [ ]:
expected_columns = ["month", "accessories", "hardware", "software"]
assert list(smart_check.columns) == expected_columns, smart_check.columns.tolist()

expected_smart_rows = [
    {"month": "2026-01", "accessories": 140.0, "hardware": 100.0, "software": 250.0},
    {"month": "2026-02", "accessories": 200.0, "hardware": 80.0, "software": 200.0},
]
assert smart_check.to_dict(orient="records") == expected_smart_rows

vanilla_mae = float(error_report.loc[error_report["stage"] == "vanilla", "pivot_mae"].iloc[0])
smart_mae = float(error_report.loc[error_report["stage"] == "smart", "pivot_mae"].iloc[0])
pivot_delta = float(quality_delta.loc[quality_delta["metric"] == "pivot_mae", "improvement"].iloc[0])

assert vanilla_mae >= smart_mae
assert pivot_delta >= 0.0

assert sorted(region_values) == ["north", "south"]
assert int(quality_issues["count"].sum()) >= 2

plan_text = report_plan.explain()
assert "derive_period" in plan_text and "summarize" in plan_text

print("quality-lift verified: smart pipeline improves or preserves pivot quality")
print("notebook-03-ok")

quality-lift verified: extra trusted table improves pivot quality
notebook-03-ok


## Part 4: Analyst Shim -> Formal Shim Translation (Ideated)

This notebook remains useful for exploratory data workflows, but the operational backlog
automation posture is moving toward a formal plan/apply/verify DSL.

The mapping idea:
- analyst quality combinators -> formal `derive` + `verify` rules
- smart joins -> formal `link` and `annotate` actions
- report tables -> formal `report` output artifacts
- heuristic repairs -> formal `causal_hypothesis` or `granger_candidate` edges until confirmed

In [1]:
from dataclasses import dataclass
from typing import Any, Dict, List

@dataclass
class FormalAction:
    kind: str
    target: str
    payload: Dict[str, Any]
    reason: str

analyst_pipeline = [
    "normalize_text(product_id, region)",
    "coerce_numeric(units, revenue)",
    "expect_domain(region in {north,south})",
    "smart_join(products)",
    "group_by(month, category)",
    "summarize(revenue=sum, units=sum)",
]

formal_translation: List[FormalAction] = [
    FormalAction(
        kind="derive",
        target="sales",
        payload={"ops": ["normalize_text", "coerce_numeric"]},
        reason="canonicalize noisy input before relational composition",
    ),
    FormalAction(
        kind="verify",
        target="sales",
        payload={"rule": "domain(region) subset {north,south}"},
        reason="quality gate before downstream aggregation",
    ),
    FormalAction(
        kind="link",
        target="#171",
        payload={"edge_kind": "correlates_with", "to": "#319"},
        reason="relational substrate and graph-theory cluster connection",
    ),
    FormalAction(
        kind="annotate",
        target="#123",
        payload={"note": "translation evidence from analyst workflow"},
        reason="audit trail for backlog governance",
    ),
    FormalAction(
        kind="report",
        target="monthly-category-summary",
        payload={"shape": ["month", "category", "revenue", "units"]},
        reason="materialized artifact for planning and review",
    ),
]

print("analyst pipeline steps:")
for step in analyst_pipeline:
    print("-", step)

print("\nformal translation actions:")
for action in formal_translation:
    print(action)

assert len(formal_translation) >= 5
assert any(a.kind == "verify" for a in formal_translation)
assert any(a.kind == "report" for a in formal_translation)

print("analyst-to-formal translation sketch verified")

analyst pipeline steps:
- normalize_text(product_id, region)
- coerce_numeric(units, revenue)
- expect_domain(region in {north,south})
- smart_join(products)
- group_by(month, category)
- summarize(revenue=sum, units=sum)

formal translation actions:
FormalAction(kind='derive', target='sales', payload={'ops': ['normalize_text', 'coerce_numeric']}, reason='canonicalize noisy input before relational composition')
FormalAction(kind='verify', target='sales', payload={'rule': 'domain(region) subset {north,south}'}, reason='quality gate before downstream aggregation')
FormalAction(kind='link', target='#171', payload={'edge_kind': 'correlates_with', 'to': '#319'}, reason='relational substrate and graph-theory cluster connection')
FormalAction(kind='annotate', target='#123', payload={'note': 'translation evidence from analyst workflow'}, reason='audit trail for backlog governance')
FormalAction(kind='report', target='monthly-category-summary', payload={'shape': ['month', 'category', 'revenue',